In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
filepath = '/content/drive/MyDrive/SwiftTraq/Portfolio/017_CreditScore/data/phase2_CreditScoring.csv'
full_df = pd.read_csv(filepath)

In [ ]:
full_df.head()

,Loanref,Credit_Score,Mortgage_Insurance,Number_of_units,CLoan_to_value,Debt_to_income,Original_amount,OLoan_to_value,OEIR,Loan_term,...,Seller_41,Seller_42,Seller_43,Seller_44,Seller_45,Seller_46,Seller_47,Seller_48,Seller_49,DFlag
0,F16Q30395024,760.0,30.0,1,95.0,39.0,237000.0,95.0,3.375,360,...,0,0,0,0,0,0,0,0,0,0
1,F16Q30138152,797.0,0.0,1,80.0,30.0,417000.0,80.0,2.625,180,...,0,0,0,0,0,0,0,0,0,0
2,F16Q40029727,766.0,0.0,1,80.0,50.0,280000.0,80.0,3.875,360,...,0,0,0,0,0,0,0,0,0,0
3,F16Q40121323,796.0,0.0,1,75.0,39.0,111000.0,75.0,3.250,180,...,0,0,0,0,0,0,0,0,0,0
4,F16Q40216810,785.0,0.0,1,80.0,44.0,220000.0,80.0,3.625,360,...,0,0,0,0,1,0,0,0,0,0


###### Quality check

In [ ]:
# 1. Drop columns with many missing values(>80%)
missing_ratio = full_df.isnull().mean()
high_missing_cols = missing_ratio[missing_ratio > 0.8].index
if len(high_missing_cols) > 0:
  full_df.drop(columns=[high_missing_cols], inplace=True)

In [ ]:
numeric_cols = full_df.select_dtypes(include=[np.number]).columns.to_list()
if full_df[numeric_cols].isnull().sum().any():
  imputer = SimpleImputer(strategy='median')
  full_df[numeric_cols] = imputer.fit_transform(full_df[numeric_cols])

In [ ]:
#2. Drop useless columns
useless_cols = ['Loanref'] # high cardinality
full_df.drop(columns=useless_cols, inplace=True)

In [ ]:
# Func to update numeric col
def update_num_cols(numeric_cols, cols_to_remove):
  numeric_cols = [col for col in numeric_cols if col not in cols_to_remove]
  print(f"✅ Updated numeric columns len: {len(numeric_cols)}")
  return numeric_cols

In [ ]:
excluded_prefix = ('is_property_state_','MSA_', 'PostalCode_', 'Seller_')
# some useless general cols
excluded_cols = [col for col in numeric_cols if col.startswith(excluded_prefix)]


In [ ]:
# Drop general cols
full_df.drop(columns=excluded_cols, inplace=True)
numeric_cols = update_num_cols(numeric_cols, excluded_cols)

✅ Updated numeric columns len: 33


In [ ]:
# Categorical cols
cat_cols = [col for col in numeric_cols if full_df[col].nunique() <= 10]
cat_cols.remove('DFlag') # remove Target variable

In [ ]:
# Get continous cols from numeric
continous_cols = [col for col in numeric_cols if full_df[col].nunique() > 10]

In [ ]:
len(full_df.columns)

33

##### Feature Scaling (only continous cols)

In [ ]:
scaler = StandardScaler()
full_df[continous_cols] = scaler.fit_transform(full_df[continous_cols])

##### Feature selection(Mutual Info)

In [ ]:
target = 'DFlag'
X = full_df.drop(columns=[target])
y = full_df[target]

mi_scores = mutual_info_classif(X, y, discrete_features='auto')
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

In [ ]:
mi_series.head()

,0
Number_of_units,0.027612
is_First_time_homeowner_No,0.019798
is_Property_type_sing,0.019062
is_Origination_channel_reta,0.017976
is_Occupancy_status_prim,0.017102


In [ ]:
top_feat = mi_series.head(20).reset_index()
top_feat.columns = ['Features', 'MI Scores']
px.bar(top_feat, x='MI Scores', y='Features',
       title='Top 20 Features by Mutual Info')
#sns.barplot(x=top_feat.values, y =top_feat.index)

In [ ]:
# Select Top features
top_features = mi_series.head(20).index.to_list()
X_selected = full_df[top_features].copy()
# Append target back
X_selected['DFlag'] = y

In [ ]:
# Final Dataset for Modeling
final_df = X_selected.copy()
filepath = '/content/drive/MyDrive/SwiftTraq/Portfolio/017_CreditScore/data/'
final_df.to_csv(filepath+'CreditScoring_modeling_data.csv')
print("✅ Dataset saved for modeling")

✅ Dataset saved for modeling
